Step 1 — Configuration / file paths

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

RUBRIC_VERSION = "v6"
VALIDATION_N = 120

LABELS = [
    "Limited GenAI applicability",
    "Augmentation-oriented",
    "Automation-oriented",
    "Indeterminate mechanism",
]

BASE_DIR = Path(
    "/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes/"
)

Human_Coder1 = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Coder1_Validation_v6.xlsx"
)

Human_Coder2 = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Coder2_Validation_v6.xlsx"
)

Human_Coder = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Coder3_Validation_v6.xlsx"
)


provisional_file = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Occupational_Task_Classification_Provisional_v6.csv"
)

provisional_task_classifications = pd.read_csv(provisional_file)

print(f"Provisional task classifications: {len(provisional_task_classifications):,}")

In [ ]:
coder1 = pd.read_excel(Human_Coder1)
coder2 = pd.read_excel(Human_Coder2)

print(coder1.shape, coder2.shape)
print(coder1.columns.tolist())
print(coder2.columns.tolist())

In [ ]:
validation = (
    coder1
    .merge(
        coder2,
        on="task_id",
        suffixes=("_c1", "_c2"),
        validate="one_to_one"
    )
    .merge(
        provisional_task_classifications[
            ["task_id", "task_statement", "provisional_label"]
        ],
        on="task_id",
        how="left",
        validate="one_to_one"
    )
)
print("Matched tasks:", len(validation))
print("Missing provisional labels:", validation["provisional_label"].isna().sum())
print(
    "Coder statements identical:",
    validation["task_statement_c1"].eq(
        validation["task_statement_c2"]
    ).sum()
)
print(
    "Current statements unchanged:",
    validation["task_statement_c1"].eq(
        validation["task_statement"]
    ).sum()
)

print("\nCoder 1 labels:")
print(coder1["coder_label"].value_counts(dropna=False))

print("\nCoder 2 labels:")
print(coder2["coder_label"].value_counts(dropna=False))

In [ ]:
human_tasks = coder1[
    ["task_id", "soc6", "task_statement"]
].copy()

current_tasks = provisional_task_classifications[
    ["task_id", "soc6", "task_statement", "provisional_label"]
].copy()

for df in [human_tasks, current_tasks]:
    df["match_key"] = (
        df["soc6"].astype(str).str.strip()
        + "|"
        + df["task_statement"]
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
    )
missing_tasks = human_tasks[
    ~human_tasks["task_id"].isin(current_tasks["task_id"])
]
statement_matches = missing_tasks.merge(
    current_tasks[
        ["task_id", "match_key", "provisional_label"]
    ],
    on="match_key",
    how="left",
    suffixes=("_old", "_current")
)
print("Missing by task ID:", len(missing_tasks))
print(
    "Recovered by SOC + task statement:",
    statement_matches["provisional_label"].notna().sum()
)
print(
    "Still unmatched:",
    statement_matches["provisional_label"].isna().sum()
)

In [ ]:
current_validation = validation[
    validation["provisional_label"].notna()
].copy()

print("Usable validation tasks:", len(current_validation))
print("SOC6 occupations represented:", current_validation["soc6_c1"].nunique())

for column in [
    "coder_label_c1",
    "coder_label_c2",
    "provisional_label"
]:
    print(f"\n{column}:")
    print(current_validation[column].value_counts())

In [ ]:
human_agreement = (
    current_validation["coder_label_c1"]
    == current_validation["coder_label_c2"]
)

print("Human–human agreements:", human_agreement.sum())
print("Human–human disagreements:", (~human_agreement).sum())
print("Agreement rate:", f"{human_agreement.mean():.1%}")

human_kappa = cohen_kappa_score(
    current_validation["coder_label_c1"],
    current_validation["coder_label_c2"]
)

print("Cohen's kappa:", f"{human_kappa:.3f}")

pd.crosstab(
    current_validation["coder_label_c1"],
    current_validation["coder_label_c2"]
)

In [ ]:
model_human_results = []

for name, column in {
    "Model vs Coder 1": "coder_label_c1",
    "Model vs Coder 2": "coder_label_c2"
}.items():

    agreement = (
        current_validation["provisional_label"]
        == current_validation[column]
    )

    model_human_results.append({
        "comparison": name,
        "n": len(current_validation),
        "agreements": agreement.sum(),
        "agreement_rate": agreement.mean(),
        "cohen_kappa": cohen_kappa_score(
            current_validation["provisional_label"],
            current_validation[column]
        )
    })

model_human_results = pd.DataFrame(model_human_results)
display(model_human_results)

In [ ]:
for name, column in {
    "Model vs Coder 1": "coder_label_c1",
    "Model vs Coder 2": "coder_label_c2"
}.items():

    print(f"\n{name}")

    matrix = pd.crosstab(
        current_validation["provisional_label"],
        current_validation[column]
    ).reindex(
        index=LABELS,
        columns=LABELS,
        fill_value=0
    )

    display(matrix)